So making a recap: doing the bootstrap on the full alignment, we find that up until the third mutation (the deleterious D189S + two compensatory mutations) the proposals are the same, then they vary a lot among them.

We would then like to use the same algorithm but only on the red sector in order to see if we can make it more robust.

In [17]:
import numpy as np
from Bio import SeqIO

In [75]:
def zero_sum_gauge(J, h):
	if J is None:
		L,q = h.shape
		J = np.zeros((L,L,q,q))
		h_zg = np.copy(h)
		h_zg -= np.expand_dims(np.mean(h,axis = 1),axis = 1) 
		h_zg += np.sum(np.mean(J,axis=3)-np.expand_dims(np.mean(J,axis=(2,3)),axis=2),axis=1)
		return None,h_zg
	
	if h is None:
		L,q = J.shape[0],J.shape[2]
		J_zg = np.copy(J)
		J_zg -= np.expand_dims(np.mean(J,axis = 2),axis = 2) 
		J_zg -= np.expand_dims(np.mean(J,axis=3),axis =3) 
		J_zg += np.expand_dims(np.mean(J,axis=(2,3)),axis=(2,3))
		return J_zg, None

	J_zg = np.copy(J)
	h_zg = np.copy(h)

	h_zg -= np.expand_dims(np.mean(h,axis = 1),axis = 1) 
	h_zg += np.sum(np.mean(J,axis=3)-np.expand_dims(np.mean(J,axis=(2,3)),axis=2),axis=1)

	J_zg -= np.expand_dims(np.mean(J,axis = 2),axis = 2) 
	J_zg -= np.expand_dims(np.mean(J,axis=3),axis =3) 
	J_zg += np.expand_dims(np.mean(J,axis=(2,3)),axis=(2,3))

	return J_zg, h_zg

def find_mutational_scan(wildtype_sequence, J, h):
	L, q = h.shape
	mut = np.zeros((q, L))
	for k in range(L):
		for b in range(q):
			delta_E = h[k, wildtype_sequence[k]] - h[k, b] - np.sum(J[k, np.arange(L), b, wildtype_sequence]) + np.sum(J[k, np.arange(L), wildtype_sequence[k], wildtype_sequence])
			mut[b, k] = delta_E
	return mut

def sequence_to_array(sequence):
    return np.array([c for c in sequence])

def aa_to_int(aa):
    alphabet = "-ACDEFGHIKLMNPQRSTVWY"
    return {aa: i for i, aa in enumerate(alphabet)}.get(aa)

def int_to_aa(aa_int):
    alphabet = "-ACDEFGHIKLMNPQRSTVWY"
    return {i: aa for i, aa in enumerate(alphabet)}.get(aa_int)

def array_aa_to_int(array_of_aa):
    return np.vectorize(aa_to_int)(array_of_aa)

def propose_mutations(
        model_filename: str,
        n_compensatory_mutations: int,
        pinned_mutations: list,
        residue_numbering_filename: str,
        alignment_filename: str,
        wildtype_sequence_accession: str):
    # load the model, get the J and h arrays
    model = np.load(model_filename, allow_pickle=True).item()
    J, h = model["J"], model["h"]
    J, h = zero_sum_gauge(J, h)
    # find the wildtype sequence in the alignment
    for record in SeqIO.parse(alignment_filename, "fasta"):
        if wildtype_sequence_accession in record.description:
            wildtype_sequence = sequence_to_array(record.seq)
            wildtype_sequence = array_aa_to_int(wildtype_sequence)
            continue
    # load residue numbering
    residue_numbering = np.load(residue_numbering_filename)
    # check that all the sequence lengths are the same
    if len({J.shape[0], J.shape[1], h.shape[0], len(wildtype_sequence), len(residue_numbering)}) > 1:
        raise ValueError("L (length of sequence) should be the same in J, h, the wildtype sequence and the residue numbering array.")
    # now pin the initial mutations
    total_mutations = []
    mutant_sequence = np.copy(wildtype_sequence)
    for pinned_mutation in pinned_mutations:
        pinned_position_string, pinned_aminoacid_string = pinned_mutation
        mutation_position = np.where(residue_numbering == pinned_position_string)[0]
        mutation_aminoacid = aa_to_int(pinned_aminoacid_string)
        mutant_sequence[mutation_position] = mutation_aminoacid
        print(residue_numbering[mutation_position], int_to_aa(mutation_aminoacid))
        total_mutations.append((mutation_position, mutation_aminoacid))
    #
    mutational_scan_wildtype = find_mutational_scan(wildtype_sequence, J, h)
    for compensatory_mutation_index in range(n_compensatory_mutations):
        mutational_scan_mutant = find_mutational_scan(mutant_sequence, J, h)
        dde = mutational_scan_mutant - mutational_scan_wildtype
        for (mutation_position, mutation_aminoacid) in total_mutations:
            dde[:, mutation_position] = np.nan
        indices_dde_minimum = np.unravel_index(np.nanargmin(dde, axis=None), dde.shape)
        mutation_aminoacid, mutation_position = indices_dde_minimum
        mutant_sequence[mutation_position] = mutation_aminoacid
        print(residue_numbering[mutation_position], int_to_aa(mutation_aminoacid))
        total_mutations.append((mutation_position, mutation_aminoacid))

In [77]:
propose_mutations(
    '../models/BootstrapModels/S1A_ModelSBM_N_chains50_N_iter400_Param_initzero_k_MCMC100000_lambda_J0.0_lambda_h0_m1_theta0.3_N_Av1_R8.npy',
    3,
    [("189", "D")],
    "../data/trypsin_numbering.npy",
    "../data/iter_aln_dedup_sp.faa",
    "TRY2_RAT"
)

['189'] D
16 -
188A W
158 A
